# AI-Powered ATS Resume Analyzer — RAG v2

**Input:** complete JD text + current resume PDF.

**Output:** dynamically extracted JD requirements, resume profile, exact/semantic/partial/missing skills, keywords, evidence, project relevance, ATS issues and recommendations.

### RAG implemented
**Indexing → Retrieval → Augmentation → Generation**

There is **no hardcoded technical-keyword dictionary**. Keywords are extracted from the actual JD.

Chroma is **in-memory** for Colab, avoiding the persistent SQLite readonly-database problem.

LaTeX generation is Phase 2.


In [1]:
!pip install -q pypdf chromadb sentence-transformers huggingface_hub scikit-learn pandas numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 788.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

In [2]:
import os, re, json, uuid
import numpy as np
from typing import List, Dict, Any, Optional
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from huggingface_hub import InferenceClient
from sklearn.metrics.pairwise import cosine_similarity
from getpass import getpass

print("Imports successful.")


Imports successful.


## 1. Hugging Face authentication

In [21]:
HF_TOKEN ="hf_YOUR_TOKEN_HERE"


if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Use a Hugging Face token beginning with hf_.")

os.environ["HF_TOKEN"] = HF_TOKEN

LLM_MODEL = "openai/gpt-oss-120b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 180
CHUNK_OVERLAP = 40
INITIAL_TOP_K = 10
FINAL_TOP_K = 8
MAX_NEW_TOKENS = 5000

embedding_model = SentenceTransformer(EMBEDDING_MODEL)
llm_client = InferenceClient(token=HF_TOKEN, provider="auto")

print("Embedding model loaded.")
print("LLM client initialized.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
LLM client initialized.


## 2. Upload resume and paste complete JD

In [22]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("Upload a resume PDF.")

resume_filename = next(iter(uploaded.keys()))

if not resume_filename.lower().endswith(".pdf"):
    raise ValueError("Resume must be a PDF.")

job_description = r'''
Job Description: Associate – Mapping Process (Google Maps Project)Position OverviewWe are seeking a detail-oriented and tech-savvy Mapping Analyst / GIS Associate to join our data operations team. In this role, you will be responsible for reviewing, analyzing, and modifying global geographic data to ensure optimal mapping experiences, accurate routing, and precise point-of-interest data for millions of global users.Job Type: Full-time, PermanentExperience: 0 to 3 years (Freshers with strong analytical skills are highly encouraged to apply)Shift Requirements: Rotational shifts (including night shifts)Key ResponsibilitiesData Verification & Quality Control: Review, edit, and validate spatial and attribute datasets (such as road networks, addresses, turn restrictions, and business locations) using specialized data tools.Map Rectification: Evaluate user-reported map modifications, feedback, and imagery to resolve navigation anomalies and improve overall map accuracy.Imagery & Logic Analysis: Interpret satellite imagery, aerial photos, and street-level views to map complex configurations like highway exits, one-way streets, and new developments.Compliance with Traffic Laws: Apply an understanding of international driving laws, traffic signs, and standard road layouts to route structuring and directional guidelines.Collaboration: Partner with global cross-functional data curation teams to meet strict project timelines and maintain high data standards.Required Skills & QualificationsEducation: Any Graduation degree completed (B.E., B.Tech, B.Sc, B.Com, or B.A.).Tech-Savviness: Exceptional familiarity with digital navigation, web applications, and productivity ecosystems (specifically Google Maps, Google Earth, and Google Docs).Communication: Excellent written and verbal English communication skills to document technical criteria and interface with international teams.Analytical Skills: Strong pattern-recognition capabilities, spatial reasoning, and sharp attention to detail.Domain Knowledge (Plus): Basic foundational knowledge of GIS (Geographic Information Systems), GPS logic, or map annotation is highly desirable but not mandatory.Preferred RequirementsFamiliarity or professional exposure to tools like ArcGIS, QGIS, or AutoCAD map layers.Strong knowledge of worldwide geography and international formatting conventions (such as address variations across countries).
'''

if "PASTE THE COMPLETE JOB DESCRIPTION HERE" in job_description:
    raise ValueError("Replace the JD placeholder with the complete JD text.")

print("Resume:", resume_filename)
print("JD characters:", len(job_description))


Saving VijayReddy_wipro.pdf to VijayReddy_wipro (1).pdf
Resume: VijayReddy_wipro (1).pdf
JD characters: 2402


## 3. PDF extraction, cleaning and chunking

In [23]:
def extract_pdf_text(path):
    reader = PdfReader(path)
    pages = []
    for page_no, page in enumerate(reader.pages, 1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append(f"[PAGE {page_no}]\n{text}")
    return "\n".join(pages)

def clean_text(text):
    return re.sub(r"\s+", " ", text.replace("\x00", " ")).strip()

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = max(end - overlap, start + 1)
    return chunks

resume_text = extract_pdf_text(resume_filename)

if not resume_text.strip():
    raise ValueError("No selectable text found. This may be a scanned PDF.")

resume_clean = clean_text(resume_text)
jd_clean = clean_text(job_description)

resume_chunks = chunk_text(resume_clean)
jd_chunks = chunk_text(jd_clean)

print("Resume chunks:", len(resume_chunks))
print("JD chunks:", len(jd_chunks))


Resume chunks: 3
JD chunks: 2


## 4. INDEXING — metadata, embeddings and in-memory Chroma

In [24]:
documents, metadatas, ids = [], [], []

for i, chunk in enumerate(resume_chunks):
    documents.append(chunk)
    metadatas.append({"source": "resume", "chunk_id": i})
    ids.append(f"resume_{i}_{uuid.uuid4().hex[:8]}")

for i, chunk in enumerate(jd_chunks):
    documents.append(chunk)
    metadatas.append({"source": "job_description", "chunk_id": i})
    ids.append(f"jd_{i}_{uuid.uuid4().hex[:8]}")

embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="ats_resume_jd"
)

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print("Indexed chunks:", collection.count())


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed chunks: 10


## 5. RETRIEVAL — vector search + multi-query reranking

In [25]:
def retrieve(query, source=None, top_k=INITIAL_TOP_K):
    q = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()

    kwargs = {
        "query_embeddings": [q],
        "n_results": min(top_k, collection.count())
    }

    if source:
        kwargs["where"] = {"source": source}

    result = collection.query(**kwargs)

    docs = result["documents"][0]
    metas = result["metadatas"][0]
    distances = result.get("distances", [[]])[0]

    return [
        {
            "text": d,
            "metadata": m,
            "distance": float(distances[i]) if distances else None
        }
        for i, (d, m) in enumerate(zip(docs, metas))
    ]

def multi_query_retrieve(queries, source, final_top_k=FINAL_TOP_K):
    candidates = []
    for q in queries:
        candidates.extend(retrieve(q, source, INITIAL_TOP_K))

    candidates = list({x["text"]: x for x in candidates}.values())

    if not candidates:
        return []

    texts = [x["text"] for x in candidates]

    q_emb = embedding_model.encode(
        queries,
        normalize_embeddings=True
    )
    d_emb = embedding_model.encode(
        texts,
        normalize_embeddings=True
    )

    scores = cosine_similarity(d_emb, q_emb).max(axis=1)

    for item, score in zip(candidates, scores):
        item["rerank_score"] = float(score)

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return candidates[:final_top_k]

JD_QUERIES = [
    "required skills and technologies",
    "preferred skills and technologies",
    "programming languages frameworks libraries tools",
    "AI machine learning deep learning NLP LLM RAG",
    "SQL databases data analytics business intelligence",
    "cloud DevOps APIs software engineering",
    "education experience qualifications certifications",
    "job responsibilities duties"
]

RESUME_QUERIES = [
    "programming languages technical skills",
    "frameworks libraries tools platforms",
    "AI machine learning deep learning NLP LLM RAG",
    "SQL databases data analytics business intelligence",
    "cloud DevOps APIs software engineering",
    "projects technologies used",
    "work experience internships responsibilities",
    "education qualifications certifications"
]

jd_retrieved = multi_query_retrieve(JD_QUERIES, "job_description")
resume_retrieved = multi_query_retrieve(RESUME_QUERIES, "resume")

print("JD retrieved:", len(jd_retrieved))
print("Resume retrieved:", len(resume_retrieved))


JD retrieved: 2
Resume retrieved: 3


## 6. AUGMENTATION — retrieved context

In [26]:
def build_context(results):
    parts = []
    for i, item in enumerate(results, 1):
        parts.append(
            f"--- Evidence {i} ---\n"
            f"Source: {item['metadata']['source']}\n"
            f"Score: {item.get('rerank_score', 0):.4f}\n\n"
            f"{item['text']}"
        )
    return "\n\n".join(parts)
jd_context = build_context(jd_retrieved)
resume_context = build_context(resume_retrieved)

print("JD context chars:", len(jd_context))
print("Resume context chars:", len(resume_context))


JD context chars: 2835
Resume context chars: 3747


In [27]:
resume_context

'--- Evidence 1 ---\nSource: resume\nScore: 0.5265\n\n[PAGE 1] Maram Vijayreddy Hyderabad, India +91-7794015284 — maramvijayreddy893@gmail.com LinkedIn — GitHub — Portfolio Professional Summary Software Engineer Aspirant with strong foundations in Python, Java, SQL, Object-Oriented Programming, Data Structures, and Software Development. Experienced in developing applications and data-driven solutions through internships and academic projects. Skilled in debugging, testing, problem-solving, SDLC practices, and collaborative software development. Technical Skills Programming:Python, Java, SQL Software Development:Object-Oriented Programming (OOP), Data Structures & Algorithms(fundamentals), SDLC, REST APIs(basics), Debugging, Testing(basics) Databases:MySQL Libraries/F rameworks:Pandas, NumPy, Scikit-learn(basics) T ools:Git, GitHub, VS Code, Jupyter Notebook, Google Colab, Power BI Education B.T ech in Computer Science (AI & Machine Learning)2022 – 2026 Nalla Narasimha Reddy Education S

## 7. LLM helper + connectivity test

In [28]:
def generate_llm(system_prompt, user_prompt, max_tokens=MAX_NEW_TOKENS):
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

test = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Reply exactly with CONNECTION_OK"}],
    temperature=0,
    max_tokens=10
)

print(test.choices[0].message.content)


None


## 8. GENERATION — dynamically extract requirements from the actual JD

In [29]:
JD_SYSTEM = (
    "You are a precise job-description information extraction system. "
    "Extract ONLY information supported by the supplied JD. "
    "Do not use a predefined keyword list. "
    "Do not invent technologies. "
    "Return valid JSON only."
)

JD_PROMPT = f"""
Extract the important requirements and keywords from this ACTUAL job description.

Return exactly:

{{
  "job_title": "",
  "required_skills": [],
  "preferred_skills": [],
  "programming_languages": [],
  "frameworks_and_libraries": [],
  "tools_and_platforms": [],
  "databases": [],
  "cloud_and_devops": [],
  "ai_ml_skills": [],
  "data_skills": [],
  "soft_skills": [],
  "education_requirements": [],
  "experience_requirements": [],
  "certifications": [],
  "important_general_keywords": []
}}

Only extract terms actually supported by the JD.

JOB DESCRIPTION:
{jd_clean}

RETRIEVED JD EVIDENCE:
{jd_context}
"""

jd_raw = generate_llm(JD_SYSTEM, JD_PROMPT, 3500)
print(jd_raw)


{
  "job_title": "Associate – Mapping Process (Google Maps Project)",
  "required_skills": [
    "Data Verification",
    "Quality Control",
    "Map Rectification",
    "Imagery & Logic Analysis",
    "Compliance with Traffic Laws",
    "Collaboration with cross‑functional teams",
    "Pattern Recognition",
    "Spatial Reasoning",
    "Attention to Detail"
  ],
  "preferred_skills": [
    "Familiarity with ArcGIS",
    "Familiarity with QGIS",
    "Familiarity with AutoCAD map layers",
    "Knowledge of worldwide geography",
    "Understanding of international address formatting conventions"
  ],
  "programming_languages": [],
  "frameworks_and_libraries": [],
  "tools_and_platforms": [
    "Google Maps",
    "Google Earth",
    "Google Docs",
    "ArcGIS",
    "QGIS",
    "AutoCAD (map layers)"
  ],
  "databases": [],
  "cloud_and_devops": [],
  "ai_ml_skills": [],
  "data_skills": [
    "Spatial data handling",
    "Attribute dataset management",
    "Satellite imagery interpretati

## 9. Parse JD profile

In [30]:
def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.I)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found.")
    return json.loads(text[start:end+1])

jd_profile = extract_json(jd_raw)

print(json.dumps(jd_profile, indent=2, ensure_ascii=False))


{
  "job_title": "Associate – Mapping Process (Google Maps Project)",
  "required_skills": [
    "Data Verification",
    "Quality Control",
    "Map Rectification",
    "Imagery & Logic Analysis",
    "Compliance with Traffic Laws",
    "Collaboration with cross‑functional teams",
    "Pattern Recognition",
    "Spatial Reasoning",
    "Attention to Detail"
  ],
  "preferred_skills": [
    "Familiarity with ArcGIS",
    "Familiarity with QGIS",
    "Familiarity with AutoCAD map layers",
    "Knowledge of worldwide geography",
    "Understanding of international address formatting conventions"
  ],
  "programming_languages": [],
  "frameworks_and_libraries": [],
  "tools_and_platforms": [
    "Google Maps",
    "Google Earth",
    "Google Docs",
    "ArcGIS",
    "QGIS",
    "AutoCAD (map layers)"
  ],
  "databases": [],
  "cloud_and_devops": [],
  "ai_ml_skills": [],
  "data_skills": [
    "Spatial data handling",
    "Attribute dataset management",
    "Satellite imagery interpretati

## 10. Build keyword set from the actual JD

In [31]:
def flatten_string_lists(data):
    values = []
    for items in data.values():
        if isinstance(items, list):
            for item in items:
                if isinstance(item, str) and item.strip():
                    values.append(item.strip())
    return sorted(set(values), key=str.lower)

jd_keywords = flatten_string_lists(jd_profile)

print("===== KEYWORDS FROM THE ACTUAL JD =====")
for x in jd_keywords:
    print("•", x)

print("Total JD terms:", len(jd_keywords))


===== KEYWORDS FROM THE ACTUAL JD =====
• 0 to 3 years of experience
• Addresses
• Aerial photo analysis
• Analytical thinking
• Any graduation degree (B.E., B.Tech, B.Sc, B.Com, B.A.)
• ArcGIS
• Attention to Detail
• Attention to detail
• Attribute dataset management
• AutoCAD (map layers)
• Business locations
• Collaboration with cross‑functional teams
• Compliance with Traffic Laws
• Data operations team
• Data Verification
• Excellent verbal English communication
• Excellent written English communication
• Familiarity with ArcGIS
• Familiarity with AutoCAD map layers
• Familiarity with QGIS
• Freshers with strong analytical skills encouraged
• Geographic Information Systems
• GIS Associate
• Global users
• Google Docs
• Google Earth
• Google Maps
• GPS logic
• Highway exits
• Imagery & Logic Analysis
• International driving laws
• Knowledge of worldwide geography
• Map annotation
• Map Rectification
• Mapping Analyst
• New developments
• Night shifts
• One‑way streets
• Pattern Rec

## 11. GENERATION — dynamically extract candidate profile from resume evidence

In [32]:
RESUME_SYSTEM = (
    "You are a precise resume information extraction system. "
    "Extract ONLY information supported by the resume evidence. "
    "Never invent skills, experience, projects, education or certifications. "
    "Return valid JSON only."
)

RESUME_PROMPT = f"""
Extract the candidate profile from the retrieved resume evidence.

Return exactly:

{{
  "programming_languages": [],
  "frameworks_and_libraries": [],
  "tools_and_platforms": [],
  "databases": [],
  "cloud_and_devops": [],
  "ai_ml_skills": [],
  "data_skills": [],
  "soft_skills": [],
  "projects": [],
  "experience": [],
  "education": [],
  "certifications": [],
  "important_keywords": []
}}

RESUME EVIDENCE:
{resume_context}
"""

resume_raw = generate_llm(RESUME_SYSTEM, RESUME_PROMPT, 4000)
resume_profile = extract_json(resume_raw)

print(json.dumps(resume_profile, indent=2, ensure_ascii=False))


{
  "programming_languages": [
    "Python",
    "Java",
    "SQL"
  ],
  "frameworks_and_libraries": [
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "FastAPI"
  ],
  "tools_and_platforms": [
    "Git",
    "GitHub",
    "VS Code",
    "Jupyter Notebook",
    "Google Colab",
    "Power BI",
    "Google Cloud"
  ],
  "databases": [
    "MySQL"
  ],
  "cloud_and_devops": [
    "Google Cloud"
  ],
  "ai_ml_skills": [
    "Machine Learning",
    "Generative AI",
    "Model Training",
    "Model Evaluation",
    "Data Preprocessing",
    "Feature Engineering"
  ],
  "data_skills": [
    "Data Cleaning",
    "Data Transformation",
    "Data Validation",
    "Data Quality Assurance",
    "SQL Querying",
    "Data Processing",
    "Data Automation",
    "Data Visualization"
  ],
  "soft_skills": [
    "Debugging",
    "Testing",
    "Problem Solving",
    "Collaboration",
    "Teamwork",
    "Project Delivery"
  ],
  "projects": [
    "Crop Recommendation System",
    "YouTube Analytics D

## 12. Semantic matching — actual JD terms vs actual resume terms

In [33]:
resume_terms = flatten_string_lists(resume_profile)

def semantic_match(jd_terms, resume_terms, threshold=0.72):
    if not jd_terms:
        return {"exact_matches": [], "semantic_matches": [], "missing_terms": [], "details": []}

    if not resume_terms:
        return {"exact_matches": [], "semantic_matches": [], "missing_terms": jd_terms, "details": []}

    jd_emb = embedding_model.encode(jd_terms, normalize_embeddings=True)
    resume_emb = embedding_model.encode(resume_terms, normalize_embeddings=True)
    sim = cosine_similarity(jd_emb, resume_emb)

    resume_lookup = {x.lower(): x for x in resume_terms}

    exact, semantic, missing, details = [], [], [], []

    for i, jd_term in enumerate(jd_terms):
        if jd_term.lower() in resume_lookup:
            exact.append(jd_term)
            details.append({
                "jd_term": jd_term,
                "resume_term": resume_lookup[jd_term.lower()],
                "match_type": "exact",
                "score": 1.0
            })
            continue

        j = int(np.argmax(sim[i]))
        score = float(sim[i, j])
        best = resume_terms[j]

        if score >= threshold:
            semantic.append({
                "jd_term": jd_term,
                "resume_term": best,
                "score": round(score, 4)
            })
            details.append({
                "jd_term": jd_term,
                "resume_term": best,
                "match_type": "semantic",
                "score": round(score, 4)
            })
        else:
            missing.append(jd_term)
            details.append({
                "jd_term": jd_term,
                "resume_term": best,
                "match_type": "missing",
                "score": round(score, 4)
            })

    return {
        "exact_matches": exact,
        "semantic_matches": semantic,
        "missing_terms": missing,
        "details": details
    }

matching = semantic_match(jd_keywords, resume_terms)

print(json.dumps(matching, indent=2, ensure_ascii=False))


{
  "exact_matches": [],
  "semantic_matches": [
    {
      "jd_term": "Data Verification",
      "resume_term": "Data Validation",
      "score": 0.7491
    },
    {
      "jd_term": "Team collaboration",
      "resume_term": "Collaboration",
      "score": 0.9149
    }
  ],
  "missing_terms": [
    "0 to 3 years of experience",
    "Addresses",
    "Aerial photo analysis",
    "Analytical thinking",
    "Any graduation degree (B.E., B.Tech, B.Sc, B.Com, B.A.)",
    "ArcGIS",
    "Attention to Detail",
    "Attention to detail",
    "Attribute dataset management",
    "AutoCAD (map layers)",
    "Business locations",
    "Collaboration with cross‑functional teams",
    "Compliance with Traffic Laws",
    "Data operations team",
    "Excellent verbal English communication",
    "Excellent written English communication",
    "Familiarity with ArcGIS",
    "Familiarity with AutoCAD map layers",
    "Familiarity with QGIS",
    "Freshers with strong analytical skills encouraged",
    "Ge

## 13. FINAL GENERATION — RAG ATS analysis

In [34]:
ATS_SYSTEM = (
    "You are an expert ATS resume analysis system. "
    "Never invent candidate information. "
    "Use retrieved evidence and extracted profiles. "
    "Distinguish exact, semantic, partial and missing. "
    "Do not encourage false claims. "
    "Return valid JSON only."
)

ATS_PROMPT = f"""
Analyze this candidate against this job description.

RETRIEVED JD EVIDENCE:
{jd_context}

RETRIEVED RESUME EVIDENCE:
{resume_context}

DYNAMIC JD PROFILE:
{json.dumps(jd_profile, indent=2, ensure_ascii=False)}

DYNAMIC RESUME PROFILE:
{json.dumps(resume_profile, indent=2, ensure_ascii=False)}

SEMANTIC MATCHING BASELINE:
{json.dumps(matching, indent=2, ensure_ascii=False)}

Return exactly:

{{
  "overall_summary": "",
  "matched_skills": [
    {{
      "skill": "",
      "match_type": "exact|semantic",
      "resume_evidence": "",
      "reason": ""
    }}
  ],
  "partial_matches": [
    {{
      "jd_requirement": "",
      "candidate_evidence": "",
      "gap": ""
    }}
  ],
  "missing_skills": [
    {{
      "skill": "",
      "importance": "high|medium|low",
      "reason": ""
    }}
  ],
  "jd_keywords": [],
  "resume_keywords": [],
  "missing_keywords": [],
  "experience_analysis": "",
  "education_analysis": "",
  "project_relevance": [
    {{
      "project": "",
      "relevance": "high|medium|low",
      "reason": ""
    }}
  ],
  "ats_issues": [],
  "recommendations": []
}}

JD keywords must come from the actual JD profile.
"""

final_raw = generate_llm(ATS_SYSTEM, ATS_PROMPT, MAX_NEW_TOKENS)
ats_result = extract_json(final_raw)

print(json.dumps(ats_result, indent=2, ensure_ascii=False))


{
  "overall_summary": "The candidate is a strong software‑engineering and data‑analytics junior with solid programming, SQL, and machine‑learning experience, but lacks direct GIS, mapping, and spatial‑data expertise required for the Associate – Mapping Process role. Only a few semantic overlaps (e.g., data validation ↔ data verification, collaboration) are present; most core mapping competencies, tools, and domain knowledge are missing.",
  "matched_skills": [
    {
      "skill": "Data Verification",
      "match_type": "semantic",
      "resume_evidence": "Data Validation (listed under data_skills)",
      "reason": "Verification of data quality is conceptually equivalent to validation of data in the resume."
    },
    {
      "skill": "Team collaboration",
      "match_type": "semantic",
      "resume_evidence": "Collaboration (listed under soft_skills)",
      "reason": "Collaboration indicates ability to work with others, aligning with the JD's collaboration requirement."
    }


In [35]:
ats_result

{'overall_summary': 'The candidate is a strong software‑engineering and data‑analytics junior with solid programming, SQL, and machine‑learning experience, but lacks direct GIS, mapping, and spatial‑data expertise required for the Associate – Mapping Process role. Only a few semantic overlaps (e.g., data validation ↔ data verification, collaboration) are present; most core mapping competencies, tools, and domain knowledge are missing.',
 'matched_skills': [{'skill': 'Data Verification',
   'match_type': 'semantic',
   'resume_evidence': 'Data Validation (listed under data_skills)',
   'reason': 'Verification of data quality is conceptually equivalent to validation of data in the resume.'},
  {'skill': 'Team collaboration',
   'match_type': 'semantic',
   'resume_evidence': 'Collaboration (listed under soft_skills)',
   'reason': "Collaboration indicates ability to work with others, aligning with the JD's collaboration requirement."}],
 'partial_matches': [{'jd_requirement': 'Collaborat

## 14. Coverage and report

In [36]:
exact_count = len(matching["exact_matches"])
semantic_count = len(matching["semantic_matches"])
total = len(jd_keywords)

exact_coverage = exact_count / total * 100 if total else 0
semantic_coverage = (exact_count + semantic_count) / total * 100 if total else 0

print("=" * 80)
print("ATS ANALYSIS")
print("=" * 80)

print("\nSUMMARY:\n", ats_result.get("overall_summary", ""))
print("\nJD TERMS:", total)
print("EXACT COVERAGE: {:.2f}%".format(exact_coverage))
print("EXACT + SEMANTIC COVERAGE: {:.2f}%".format(semantic_coverage))

for title, key in [
    ("MATCHED SKILLS", "matched_skills"),
    ("PARTIAL MATCHES", "partial_matches"),
    ("MISSING SKILLS", "missing_skills"),
    ("JD KEYWORDS", "jd_keywords"),
    ("RESUME KEYWORDS", "resume_keywords"),
    ("MISSING KEYWORDS", "missing_keywords"),
    ("PROJECT RELEVANCE", "project_relevance"),
    ("ATS ISSUES", "ats_issues"),
    ("RECOMMENDATIONS", "recommendations")
]:
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    for item in ats_result.get(key, []):
        print(json.dumps(item, indent=2, ensure_ascii=False) if isinstance(item, dict) else "• " + str(item))

print("\nEXPERIENCE ANALYSIS:\n", ats_result.get("experience_analysis", ""))
print("\nEDUCATION ANALYSIS:\n", ats_result.get("education_analysis", ""))


ATS ANALYSIS

SUMMARY:
 The candidate is a strong software‑engineering and data‑analytics junior with solid programming, SQL, and machine‑learning experience, but lacks direct GIS, mapping, and spatial‑data expertise required for the Associate – Mapping Process role. Only a few semantic overlaps (e.g., data validation ↔ data verification, collaboration) are present; most core mapping competencies, tools, and domain knowledge are missing.

JD TERMS: 51
EXACT COVERAGE: 0.00%
EXACT + SEMANTIC COVERAGE: 3.92%

MATCHED SKILLS
{
  "skill": "Data Verification",
  "match_type": "semantic",
  "resume_evidence": "Data Validation (listed under data_skills)",
  "reason": "Verification of data quality is conceptually equivalent to validation of data in the resume."
}
{
  "skill": "Team collaboration",
  "match_type": "semantic",
  "resume_evidence": "Collaboration (listed under soft_skills)",
  "reason": "Collaboration indicates ability to work with others, aligning with the JD's collaboration requ

## 15. Save JSON for Phase 2

In [37]:
output = {
    "model": LLM_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "jd_profile": jd_profile,
    "resume_profile": resume_profile,
    "semantic_matching": matching,
    "coverage": {
        "jd_terms": total,
        "exact_percent": exact_coverage,
        "exact_plus_semantic_percent": semantic_coverage
    },
    "ats_analysis": ats_result
}

output_path = "/content/ats_analysis_v2.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Saved:", output_path)

from google.colab import files
files.download(output_path)


Saved: /content/ats_analysis_v2.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Architecture

```text
JD text + Resume PDF
        ↓
Extraction + Cleaning
        ↓
Chunking + Metadata
        ↓
Embedding
        ↓
Chroma Indexing
        ↓
Multi-query Retrieval
        ↓
Cosine Reranking
        ↓
Retrieved Context
        ↓
Prompt Augmentation
        ↓
Dynamic JD Requirement Extraction
        +
Dynamic Resume Profile Extraction
        ↓
Semantic Matching
        ↓
Final RAG Generation
        ↓
Structured ATS Analysis
        ↓
Phase 2: LaTeX Generator
```

**The keyword set is derived from the user's actual JD. No fixed technical-keyword dictionary is used.**
